# Locy Use Case: Supply Chain Provenance

Trace multi-hop upstream supplier lineage for a finished component.

This notebook uses **schema-first mode** (recommended): labels, edge types, and typed properties are defined before ingest.


## How To Read This Notebook

- Step 1 initializes an isolated local database.
- Step 2 defines schema (the recommended production path).
- Step 3 seeds a minimal graph for this use case.
- Step 4 declares Locy rules and query statements.
- Steps 5-6 evaluate and inspect command/query outputs.
- Step 7 tells you what to look for in the results.


## 1) Setup

Creates a temporary database directory so the example is reproducible and leaves no state behind.


In [ ]:
import os
import shutil
import tempfile
from pprint import pprint

import uni_db

DB_DIR = tempfile.mkdtemp(prefix="uni_locy_")
print("DB_DIR:", DB_DIR)

db = uni_db.Database(DB_DIR)


## 2) Define Schema (Recommended)

Define labels, property types, and edge types before inserting data.


In [ ]:
(
    db.schema()
    .label("Part")
        .property("sku", "string")
        .property("kind", "string")
    .done()
    .edge_type("SOURCED_FROM", ["Part"], ["Part"])
    .done()
    .apply()
)

print('Schema created')


## 3) Seed Graph Data

Insert only the entities/relationships needed for this scenario so rule behavior stays easy to inspect.


In [ ]:
db.execute("CREATE (:Part {sku: 'C1', kind: 'finished'})")
db.execute("CREATE (:Part {sku: 'B1', kind: 'subassembly'})")
db.execute("CREATE (:Part {sku: 'B2', kind: 'subassembly'})")
db.execute("CREATE (:Part {sku: 'R1', kind: 'raw'})")
db.execute("CREATE (:Part {sku: 'R2', kind: 'raw'})")
db.execute("MATCH (c:Part {sku:'C1'}), (b1:Part {sku:'B1'}) CREATE (c)-[:SOURCED_FROM]->(b1)")
db.execute("MATCH (c:Part {sku:'C1'}), (b2:Part {sku:'B2'}) CREATE (c)-[:SOURCED_FROM]->(b2)")
db.execute("MATCH (b1:Part {sku:'B1'}), (r1:Part {sku:'R1'}) CREATE (b1)-[:SOURCED_FROM]->(r1)")
db.execute("MATCH (b2:Part {sku:'B2'}), (r2:Part {sku:'R2'}) CREATE (b2)-[:SOURCED_FROM]->(r2)")
print('Seeded graph data')


## 4) Locy Program

`CREATE RULE` defines derived relations. `QUERY ... WHERE ... RETURN ...` reads from those relations.


In [ ]:
program = r'''
CREATE RULE upstream AS
MATCH (a:Part)-[:SOURCED_FROM]->(b:Part)
YIELD KEY a, KEY b

CREATE RULE upstream AS
MATCH (a:Part)-[:SOURCED_FROM]->(mid:Part)
WHERE mid IS upstream TO b
YIELD KEY a, KEY b

QUERY upstream WHERE a.sku = 'C1' RETURN b.sku AS supplier_sku, b.kind AS supplier_kind
'''
print(program)


## 5) Evaluate Locy Program

Run the program, then inspect materialization stats (`iterations`, `strata`, and executed queries).


In [ ]:
out = db.locy_evaluate(program)

print("Derived relations:", list(out["derived"].keys()))
stats = out["stats"]
print("Iterations:", stats.total_iterations)
print("Strata:", stats.strata_evaluated)
print("Queries executed:", stats.queries_executed)


## 6) Inspect Command Results

Each command result can contain `rows`; this is the easiest way to verify your rule outputs and query projections.


In [ ]:
print("Derived relation snapshots:")
for rel_name, rel_rows in out["derived"].items():
    print(f"\\n{rel_name}: {len(rel_rows)} row(s)")
    pprint(rel_rows)

if out["command_results"]:
    print("\\nCommand results:")
for i, cmd in enumerate(out["command_results"], start=1):
    print(f"\\nCommand #{i}:", cmd.get("type"))
    rows = cmd.get("rows")
    if rows is not None:
        pprint(rows)
if not out["command_results"]:
    print("\\nNo QUERY/EXPLAIN/ABDUCE command outputs in this program.")


## 7) What To Expect

Use these checks to validate output after evaluation:
- For `C1`, output should include both subassemblies (`B1`, `B2`) and raw parts (`R1`, `R2`).
- `supplier_kind` helps separate immediate suppliers vs deeper upstream tiers.
- This same pattern scales to provenance and recall workflows.


## 8) Cleanup

Delete the temporary database directory created in setup.


In [ ]:
shutil.rmtree(DB_DIR, ignore_errors=True)
print("Cleaned up", DB_DIR)
